# Лабораторная работа 2 (часть 2)

## Прикладные задачи на данных интернет-магазина

**ФИО:** Аюпова В.Р.
**Группа:** ЕТ-212
**Дата:** 02.10.2026

**Инструменты:** Python, NumPy, Pandas, Matplotlib

## Часть 0. Подготовка данных

Сформируем DataFrame на 3000 заказов интернет-магазина. 
Данные сгенерируем программно с помощью NumPy и Pandas.

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(2024)

### Справочники

Определим списки городов, категорий, статусов, способов оплаты и названий товаров.

In [2]:
cities = ["Москва", "Санкт-Петербург", "Казань", "Екатеринбург", "Челябинск"]

categories = ["Электроника", "Одежда", "Дом", "Красота", "Спорт"]

statuses = ["Доставлен", "Отменён", "Возврат"]

payments = ["Карта", "Онлайн", "Наличные"]

product_names = [
    "Смартфон", "Ноутбук", "Наушники", "Телевизор", "Планшет",
    "Футболка", "Джинсы", "Куртка", "Платье", "Кроссовки",
    "Пылесос", "Кофемашина", "Микроволновка", "Чайник", "Утюг",
    "Крем для лица", "Шампунь", "Помада", "Духи", "Тушь",
    "Гантели", "Коврик для йоги", "Скакалка", "Велосипед", "Мяч"
]

### Генерация данных

Создадим 3000 заказов. Для каждого:
- случайный клиент (customer_id) и город;
- случайный возраст (с ~30 пропусками);
- флаг VIP;
- случайный товар и категория;
- количество, цена, скидка, себестоимость;
- статус, способ оплаты;
- дата заказа в пределах 2024 года.

In [3]:
n = 3000

# Идентификатор заказа
order_id = np.arange(1, n + 1)

# Клиенты: 500 уникальных customer_id, у каждого свой город
customer_ids = np.random.randint(1, 501, size=n)
city_by_customer = {
    cid: np.random.choice(cities) 
    for cid in range(1, 501)
}
city = np.array([city_by_customer[cid] for cid in customer_ids])

# Возраст: нормальное распределение ~35 лет, стандартное отклонение 12
age = np.random.normal(loc=35, scale=12, size=n).astype(int)
age = np.clip(age, 16, 90)  # ограничим разумными рамками

# Флаг VIP: около 15% клиентов
is_vip = np.random.choice([True, False], size=n, p=[0.15, 0.85])

# Товары: случайные названия и категории
product_idx = np.random.randint(0, len(product_names), size=n)
product_name = np.array([product_names[i] for i in product_idx])

# Каждая категория привязана к своим товарам
product_to_category = {
    "Смартфон": "Электроника", "Ноутбук": "Электроника", "Наушники": "Электроника",
    "Телевизор": "Электроника", "Планшет": "Электроника",
    "Футболка": "Одежда", "Джинсы": "Одежда", "Куртка": "Одежда",
    "Платье": "Одежда", "Кроссовки": "Одежда",
    "Пылесос": "Дом", "Кофемашина": "Дом", "Микроволновка": "Дом",
    "Чайник": "Дом", "Утюг": "Дом",
    "Крем для лица": "Красота", "Шампунь": "Красота", "Помада": "Красота",
    "Духи": "Красота", "Тушь": "Красота",
    "Гантели": "Спорт", "Коврик для йоги": "Спорт", "Скакалка": "Спорт",
    "Велосипед": "Спорт", "Мяч": "Спорт"
}
category = np.array([product_to_category[p] for p in product_name])

# Количество товаров в заказе: 1-5
quantity = np.random.randint(1, 6, size=n)

# Цена: логнормальное распределение даёт реалистичные «хвосты»
price = np.round(np.random.lognormal(mean=8, sigma=0.8, size=n), 2)
price = np.clip(price, 100, 200000)

# Скидка: от 0% до 40%
discount = np.round(np.random.uniform(0, 0.4, size=n), 2)

# Себестоимость: 50-80% от цены
cost = np.round(price * np.random.uniform(0.5, 0.8, size=n) * quantity, 2)

# Статус заказа: 85% доставлено, 8% отменено, 7% возврат
status = np.random.choice(statuses, size=n, p=[0.85, 0.08, 0.07])

# Способ оплаты
payment = np.random.choice(payments, size=n)

# Дата заказа: 365 дней 2024 года
start_date = pd.Timestamp("2024-01-01")
random_days = np.random.randint(0, 366, size=n)
order_date = start_date + pd.to_timedelta(random_days, unit="D")

# Вносим ~30 пропусков в age
age_with_nan = age.astype(float)
nan_indices = np.random.choice(n, size=30, replace=False)
age_with_nan[nan_indices] = np.nan

In [4]:
df = pd.DataFrame({
    "order_id": order_id,
    "order_date": order_date,
    "customer_id": customer_ids,
    "city": city,
    "age": age_with_nan,
    "is_vip": is_vip,
    "product_name": product_name,
    "category": category,
    "quantity": quantity,
    "price": price,
    "discount": discount,
    "cost": cost,
    "status": status,
    "payment": payment
})

print(df.head())
print("\nРазмер:", df.shape)

   order_id order_date  customer_id             city   age  is_vip  \
0         1 2024-10-28          137  Санкт-Петербург  38.0   False   
1         2 2024-03-21           97           Москва   NaN   False   
2         3 2024-08-20          129        Челябинск  52.0   False   
3         4 2024-01-21           28           Москва  21.0   False   
4         5 2024-07-15           37        Челябинск  49.0    True   

  product_name     category  quantity    price  discount     cost     status  \
0    Велосипед        Спорт         3  1868.80      0.01  3566.77  Доставлен   
1      Планшет  Электроника         2   313.19      0.37   404.26  Доставлен   
2      Пылесос          Дом         2  6176.64      0.20  6734.34  Доставлен   
3    Велосипед        Спорт         1  2161.07      0.27  1229.19  Доставлен   
4      Гантели        Спорт         1   971.88      0.27   616.26  Доставлен   

    payment  
0     Карта  
1     Карта  
2     Карта  
3  Наличные  
4     Карта  

Размер: (3000

In [5]:
print("Пропусков в age:", df["age"].isna().sum())

Пропусков в age: 30


### Вывод

Сформирован DataFrame на 3000 заказов со всеми требуемыми полями:
- 14 столбцов: order_id, order_date, customer_id, city, age, is_vip, 
  product_name, category, quantity, price, discount, cost, status, payment;
- 500 уникальных клиентов, 5 городов, 5 категорий, 3 статуса, 3 способа оплаты;
- ~30 пропущенных значений в столбце age (для отработки обработки пропусков);
- даты заказов — случайные в пределах 2024 года.

## Часть 1. Первичный анализ данных

Проверим, что данные загрузились корректно и имеют ожидаемую структуру.

In [6]:
df.shape

(3000, 14)

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   order_id      3000 non-null   int64         
 1   order_date    3000 non-null   datetime64[ns]
 2   customer_id   3000 non-null   int32         
 3   city          3000 non-null   object        
 4   age           2970 non-null   float64       
 5   is_vip        3000 non-null   bool          
 6   product_name  3000 non-null   object        
 7   category      3000 non-null   object        
 8   quantity      3000 non-null   int32         
 9   price         3000 non-null   float64       
 10  discount      3000 non-null   float64       
 11  cost          3000 non-null   float64       
 12  status        3000 non-null   object        
 13  payment       3000 non-null   object        
dtypes: bool(1), datetime64[ns](1), float64(4), int32(2), int64(1), object(5)
memory usage: 2

In [8]:
df.head()

,order_id,order_date,customer_id,city,age,is_vip,product_name,category,quantity,price,discount,cost,status,payment
0,1,2024-10-28,137,Санкт-Петербург,38.0,False,Велосипед,Спорт,3,1868.80,0.01,3566.77,Доставлен,Карта
1,2,2024-03-21,97,Москва,NaN,False,Планшет,Электроника,2,313.19,0.37,404.26,Доставлен,Карта
2,3,2024-08-20,129,Челябинск,52.0,False,Пылесос,Дом,2,6176.64,0.20,6734.34,Доставлен,Карта
3,4,2024-01-21,28,Москва,21.0,False,Велосипед,Спорт,1,2161.07,0.27,1229.19,Доставлен,Наличные
4,5,2024-07-15,37,Челябинск,49.0,True,Гантели,Спорт,1,971.88,0.27,616.26,Доставлен,Карта


In [9]:
df.tail()

,order_id,order_date,customer_id,city,age,is_vip,product_name,category,quantity,price,discount,cost,status,payment
2995,2996,2024-11-07,330,Санкт-Петербург,36.0,False,Кофемашина,Дом,4,1114.12,0.20,3385.73,Доставлен,Онлайн
2996,2997,2024-01-30,443,Екатеринбург,29.0,False,Смартфон,Электроника,5,5334.23,0.30,19471.74,Доставлен,Наличные
2997,2998,2024-06-20,474,Москва,16.0,False,Духи,Красота,4,1028.18,0.25,3180.63,Доставлен,Онлайн
2998,2999,2024-05-28,151,Челябинск,28.0,False,Коврик для йоги,Спорт,4,9404.99,0.15,25241.00,Доставлен,Карта
2999,3000,2024-09-22,161,Екатеринбург,26.0,True,Микроволновка,Дом,3,4946.52,0.39,11439.89,Доставлен,Карта


In [10]:
df.describe()

,order_id,order_date,customer_id,age,quantity,price,discount,cost
count,3000.000000,3000,3000.000000,2970.000000,3000.000000,3000.000000,3000.00000,3000.000000
mean,1500.500000,2024-07-03 03:12:28.800000256,244.670333,34.730976,3.030000,4067.940667,0.20050,8055.763400
min,1.000000,2024-01-01 00:00:00,1.000000,16.000000,1.000000,204.830000,0.00000,170.370000
25%,750.750000,2024-04-04 00:00:00,113.000000,27.000000,2.000000,1722.167500,0.10000,2570.177500
50%,1500.500000,2024-07-02 00:00:00,244.000000,34.000000,3.000000,2956.780000,0.20000,5113.590000
75%,2250.250000,2024-10-04 00:00:00,371.000000,42.000000,4.000000,5017.322500,0.30000,10027.812500
max,3000.000000,2024-12-31 00:00:00,500.000000,76.000000,5.000000,50477.600000,0.40000,188108.930000
std,866.169729,NaN,146.828896,11.354955,1.429142,3882.749655,0.11651,9771.955003


In [11]:
df.isna().sum()

order_id         0
order_date       0
customer_id      0
city             0
age             30
is_vip           0
product_name     0
category         0
quantity         0
price            0
discount         0
cost             0
status           0
payment          0
dtype: int64

### Ответы на вопросы

1. **Сколько строк и столбцов в DataFrame?**
   3000 строк, 14 столбцов.

2. **В каких столбцах есть пропуски?**
   Только в `age`.

3. **Сколько пропусков в `age`?**
   30 пропусков.

4. **Сколько уникальных клиентов?**
   500 (`customer_id` от 1 до 500).

5. **Сколько уникальных товаров?**
   25 названий (см. `product_names`).

6. **Какие города представлены в данных?**
   Москва, Санкт-Петербург, Казань, Екатеринбург, Челябинск.

7. **Какие категории товаров есть?**
   Электроника, Одежда, Дом, Красота, Спорт.

8. **Какие способы оплаты используются?**
   Карта, Онлайн, Наличные.

## Часть 2. Подготовка данных

Приведём данные к рабочему виду: преобразуем даты, рассчитаем выручку 
и прибыль, определим реальные продажи, обработаем пропуски в возрасте 
и создадим возрастные группы.

### 2.1. Работа с датой

Столбец `order_date` у нас уже имеет тип `datetime64`, но для единообразия 
с методичкой выполним явное преобразование через `pd.to_datetime`. 
Затем извлечём год и месяц в отдельные столбцы.

In [4]:
df["order_date"] = pd.to_datetime(df["order_date"])

df["year"] = df["order_date"].dt.year
df["month"] = df["order_date"].dt.month

df[["order_date", "year", "month"]].head()

NameError: name 'df' is not defined